# 頚椎学習曲線 — SmallUNet 訓練

5-fold CV × 6段階の訓練サイズでSmallUNetを訓練し、学習曲線データを収集する。

**事前準備**: `00_prepare_folds.ipynb` で `cervical_folds.json` を生成しておくこと。

**Colab Pro GPU ランタイム推奨**（全30ジョブで2〜4時間程度）。

In [ ]:
# ===== 設定（ここを変えて実行する）=====
FOLDS = [1, 2, 3, 4, 5]            # 実行するfold（途中再開も可）
SIZES = [20, 40, 80, 160, 220, 259] # 訓練サイズ（maxは全データの4/5 ≈ 259）

AUGMENT = True
LOSS    = 'awl'
SIGMA   = 15.0
# ========================================

DRIVE_DATA_DIR = '/content/drive/MyDrive/spine_data/omuro_cervical'
DRIVE_LC_DIR   = '/content/drive/MyDrive/spine_data/omuro_cervical_lc'
EPOCHS     = 100
BATCH_SIZE = 8
LANDMARKS  = 'C2_center,C2_ant,C2_post,C7_sup_post,C7_inf_ant,C7_inf_post,T1_ant,T1_post'

VARIANT = f'smallunet_aug{int(AUGMENT)}_{LOSS}_s{int(SIGMA)}'
print(f'variant: {VARIANT}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
REPO=/content/spine-measure-assist
if [ ! -d "$REPO" ]; then
  git clone https://github.com/masaki39/spine-measure-assist.git "$REPO"
fi
cd "$REPO" && git pull
pip install uv -q
cd "$REPO" && uv sync --extra ml -q

In [ ]:
import json, math, os, random, shutil, subprocess, sys
import numpy as np
import onnxruntime as ort
import torch

print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

REPO = '/content/spine-measure-assist'
sys.path.insert(0, f'{REPO}/train')
from dataset import _percentile_clip_norm, _resize_with_padding

with open(os.path.join(DRIVE_LC_DIR, 'cervical_folds.json')) as f:
    fold_data = json.load(f)

LANDMARK_ORDER = LANDMARKS.split(',')

def preprocess(img_np, resize=(512, 512)):
    if img_np.ndim == 3: img_np = img_np[0]
    img_np = _percentile_clip_norm(img_np)
    t = torch.from_numpy(img_np).unsqueeze(0)
    t, scale, pad_x, pad_y = _resize_with_padding(t, resize)
    return t.unsqueeze(0), scale, pad_x, pad_y

def postprocess(hm):
    hm = hm[0]
    return [(float(np.unravel_index(np.argmax(c), c.shape)[1]),
             float(np.unravel_index(np.argmax(c), c.shape)[0])) for c in hm]

def run_subprocess(cmd, cwd):
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if result.returncode != 0:
        print('=== STDOUT ==='); print(result.stdout[-3000:])
        print('=== STDERR ==='); print(result.stderr[-3000:])
        raise RuntimeError(f'Command failed (exit {result.returncode}): {" ".join(cmd)}')
    lines = (result.stdout + result.stderr).strip().split('\n')
    print('\n'.join(lines[-5:]))

# 頚椎角度計算（eval_cervical.py と同一ロジック）
def _vec(a, b): return (b[0]-a[0], b[1]-a[1])
def _signed_slope(v):
    ang = math.degrees(math.atan2(v[1], v[0]))
    if ang > 90: ang -= 180
    elif ang < -90: ang += 180
    return -ang
def _wrap(a):
    while a > 180: a -= 360
    while a < -180: a += 360
    return a

def compute_cervical_angles(pts, spacing_mm=1.0):
    required = LANDMARK_ORDER
    if not all(k in pts for k in required): return None
    v_C2  = _vec(pts['C2_ant'],     pts['C2_post'])
    v_C7i = _vec(pts['C7_inf_ant'], pts['C7_inf_post'])
    v_T1  = _vec(pts['T1_ant'],     pts['T1_post'])
    c2c7 = _wrap(_signed_slope(v_C7i) - _signed_slope(v_C2))
    t1s  = _signed_slope(v_T1)
    sva  = (pts['C2_center'][0] - pts['C7_sup_post'][0]) * spacing_mm
    return {'C2C7_angle': c2c7, 'T1S': t1s, 'C2C7_SVA': sva}

print('Setup complete')

In [ ]:
results_dir = os.path.join(DRIVE_LC_DIR, 'results', VARIANT)
os.makedirs(results_dir, exist_ok=True)

for FOLD in FOLDS:
    print('=' * 60)
    print(f'FOLD {FOLD}  ({FOLDS.index(FOLD)+1}/{len(FOLDS)})  VARIANT={VARIANT}')
    print('=' * 60)

    test_ids      = fold_data['folds'][str(FOLD)]
    all_train_ids = [cid for k, ids in fold_data['folds'].items()
                     for cid in ids if k != str(FOLD)]

    LOCAL_TEST = f'/content/test_data_fold{FOLD}'
    if os.path.exists(LOCAL_TEST): shutil.rmtree(LOCAL_TEST)
    os.makedirs(LOCAL_TEST)
    for cid in test_ids:
        for ext in ['_image.npy', '_landmarks.json']:
            shutil.copy(os.path.join(DRIVE_DATA_DIR, cid + ext), LOCAL_TEST)
    print(f'  test={len(test_ids)} cases')

    for SIZE in SIZES:
        actual_size = min(SIZE, len(all_train_ids))
        out_path = os.path.join(results_dir, f'fold{FOLD}_size{actual_size:03d}.json')
        if os.path.exists(out_path):
            print(f'[skip] fold{FOLD}_size{actual_size:03d}')
            continue

        print(f'--- FOLD={FOLD}  SIZE={actual_size} ---')

        random.seed(FOLD * 1000 + SIZE)
        train_ids = sorted(all_train_ids)
        random.shuffle(train_ids)
        train_ids = train_ids[:actual_size]

        LOCAL_TRAIN = '/content/train_data'
        if os.path.exists(LOCAL_TRAIN): shutil.rmtree(LOCAL_TRAIN)
        os.makedirs(LOCAL_TRAIN)
        for cid in train_ids:
            for ext in ['_image.npy', '_landmarks.json']:
                shutil.copy(os.path.join(DRIVE_DATA_DIR, cid + ext), LOCAL_TRAIN)
        print(f'  train={len(train_ids)}')

        SAVE_DIR = f'/content/runs/{VARIANT}/fold{FOLD}_size{actual_size:03d}'
        cmd = [
            'uv', 'run', 'python', 'train.py',
            '--data-dir', LOCAL_TRAIN,
            '--save-dir', SAVE_DIR,
            '--landmarks', LANDMARKS,
            '--backbone', 'smallunet',
            '--sigma', str(SIGMA),
            '--loss', LOSS,
            '--epochs', str(EPOCHS),
            '--batch-size', str(BATCH_SIZE),
            '--split-seed', str(FOLD * 7 + SIZE),
        ]
        if AUGMENT:
            cmd.append('--augment')
        run_subprocess(cmd, cwd=REPO)

        ONNX_PATH = f'{SAVE_DIR}/cervical_lc.onnx'
        run_subprocess([
            'uv', 'run', 'python', 'train/export_lumbar.py',
            '--checkpoint', f'{SAVE_DIR}/best.pt',
            '--output', ONNX_PATH,
        ], cwd=REPO)

        sess = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
        all_errors = {k: [] for k in LANDMARK_ORDER}
        per_case_mres = []
        angle_errs = {'C2C7_angle': [], 'T1S': [], 'C2C7_SVA': []}

        for cid in test_ids:
            img_np = np.load(os.path.join(LOCAL_TEST, cid + '_image.npy'))
            inp_t, scale, pad_x, pad_y = preprocess(img_np)
            ort_out = sess.run(None, {'image': inp_t.numpy()})
            pred = postprocess(ort_out[0])

            with open(os.path.join(LOCAL_TEST, cid + '_landmarks.json')) as fp:
                meta = json.load(fp)
            spacing = meta.get('metadata', {}).get('spacing', [1.0])[0]
            lm = meta['landmarks_ijk']

            pred_ij = {}
            for (px, py), name in zip(pred, LANDMARK_ORDER):
                gx, gy = lm[name]['i'], lm[name]['j']
                x_o = (px - pad_x) / scale
                y_o = (py - pad_y) / scale
                err = math.sqrt((x_o - gx)**2 + (y_o - gy)**2) * spacing
                all_errors[name].append(err)
                pred_ij[name] = (x_o, y_o)

            case_mre = sum(all_errors[nm][-1] for nm in LANDMARK_ORDER) / len(LANDMARK_ORDER)
            per_case_mres.append(case_mre)

            gt_ang = meta.get('angles_deg', {})
            ai_ang = compute_cervical_angles(pred_ij, spacing_mm=spacing)
            if ai_ang and gt_ang:
                for a in angle_errs:
                    if a in ai_ang and a in gt_ang:
                        angle_errs[a].append(abs(ai_ang[a] - gt_ang[a]))

        # outlier detection (>10mm)
        is_outlier = [max(all_errors[k][i] for k in LANDMARK_ORDER) > 10.0
                      for i in range(len(test_ids))]
        n_outliers = sum(is_outlier)
        ok_idx = [i for i, o in enumerate(is_outlier) if not o]

        all_vals = [all_errors[k][i] for i in range(len(test_ids)) for k in LANDMARK_ORDER]
        ok_vals  = [all_errors[k][i] for i in ok_idx for k in LANDMARK_ORDER]

        results = {
            'fold': FOLD, 'size': actual_size, 'variant': VARIANT,
            'n_test': len(test_ids),
            'overall': {
                'mre_mm': sum(all_vals) / len(all_vals),
                'sdr2': sum(1 for e in all_vals if e <= 2.0) / len(all_vals) * 100,
                'sdr4': sum(1 for e in all_vals if e <= 4.0) / len(all_vals) * 100,
                'angle_mae': {a: sum(v)/len(v) for a, v in angle_errs.items() if v},
            },
            'overall_excl_outliers': {
                'n': len(ok_idx), 'n_outliers': n_outliers,
                'mre_mm': sum(ok_vals)/len(ok_vals) if ok_vals else None,
                'angle_mae': {a: sum(v[i] for i in ok_idx if i<len(v))/len([i for i in ok_idx if i<len(v)])
                              for a, v in angle_errs.items() if any(i<len(v) for i in ok_idx)},
            },
            'landmarks': {k: {'mre_mm': sum(all_errors[k])/len(all_errors[k]),
                               'sdr2': sum(1 for e in all_errors[k] if e<=2.0)/len(all_errors[k])*100,
                               'sdr4': sum(1 for e in all_errors[k] if e<=4.0)/len(all_errors[k])*100}
                          for k in LANDMARK_ORDER},
        }
        with open(out_path, 'w') as fp:
            json.dump(results, fp, indent=2)
        print(f"  MRE={results['overall']['mre_mm']:.2f}mm  SDR@4={results['overall']['sdr4']:.1f}%  -> saved")

print('全fold・全サイズ完了')